In [ ]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1521

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


In [1]:
# Run this if you don't have the libraries installed
# !pip install pyspark duckdb apache-airflow
# !wget download.microsoft.com
!pip install apache-airflow

  Obtaining dependency information for apache-airflow from https://files.pythonhosted.org/packages/e4/4e/430de7acec52b97ac960b28bd20645540015779fb258a375e86e39d6cdb4/apache_airflow-3.1.5-py3-none-any.whl.metadata
  Obtaining dependency information for apache-airflow-core==3.1.5 from https://files.pythonhosted.org/packages/54/fe/a665e6834244e040c6e4ea44a787111029f889058a67be057ba6dfb69a29/apache_airflow_core-3.1.5-py3-none-any.whl.metadata
  Obtaining dependency information for apache-airflow-task-sdk==1.1.5 from https://files.pythonhosted.org/packages/4c/6e/657919d3ddbd62e9857bbce43fac803102a471b6eb60a6fbfafda0605c6c/apache_airflow_task_sdk-1.1.5-py3-none-any.whl.metadata
  Obtaining dependency information for a2wsgi>=1.10.8 from https://files.pythonhosted.org/packages/02/d5/349aba3dc421e73cbd4958c0ce0a4f1aa3a738bc0d7de75d2f40ed43a535/a2wsgi-1.10.10-py3-none-any.whl.metadata
  Obtaining dependency information for aiosqlite>=0.20.0 from https://files.pythonhosted.org/packages/00/b7/e3bf


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os, sys, findspark

# 1. Setup paths
os.environ["SPARK_HOME"] = "C:/spark"
os.environ["HADOOP_HOME"] = "C:/hadoop"
os.environ["JAVA_HOME"] = r"C:\Program Files\BellSoft\LibericaJDK-17"
#os.environ["JAVA_HOME"] = "C:/Program Files/Java/jdk-21" # Verify this path!

# 2. Initialize
findspark.init()

# 3. Initialize findspark so it uses C:/spark
findspark.init()
print("Findspark initialized successfully.") 

Findspark initialized successfully.


In [9]:
import duckdb
from pyspark.sql import SparkSession
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta
import time

# Configuration
SQL_SERVER_CONFIG = {
    "url": "jdbc:sqlserver://localhost:1433;databaseName=AdventureSales;encrypt=true;trustServerCertificate=true;",
    "user": "sa",
    "password": "D@1ry@2014",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

DUCKDB_PATH = 'c:\\duckDB\\adventuresales.duckdb'

# List of tables to migrate
TABLES_TO_COPY = [
    "sttwk", "customerdist_1910a", "product", "prodcat", 
    "branch", "stockpoint", "discount_b", "discount_potTunai",
    "sls_chlprn", "slsmanexc", "slsexc_chlprn6", "dw_ilo_18", 
    "dw_ilo_oth", "slsman"
]


C:\Users\HYPE AMD\AppData\Local\Temp\ipykernel_22012\3226831316.py:4 DeprecatedImportWarning: The `airflow.operators.python.PythonOperator` attribute is deprecated. Please use `'airflow.providers.standard.operators.python.PythonOperator'`.

In [11]:

def migrate_table(table_name):
    start_time = time.time()
    spark = SparkSession.builder \
        .appName(f"Migrate_{table_name}") \
        .config("spark.jars", "C:\\jdbc\\mssql-jdbc-12.2.0.jre11.jar") \
        .getOrCreate()

    # Default reader settings
    reader = spark.read.format("jdbc") \
        .option("url", SQL_SERVER_CONFIG["url"]) \
        .option("user", SQL_SERVER_CONFIG["user"]) \
        .option("password", SQL_SERVER_CONFIG["password"]) \
        .option("driver", SQL_SERVER_CONFIG["driver"]) \
        .option("batchsize", "100000") # Great for keeping memory stable

    if table_name.lower() == 'sttwk':
        # Your custom subquery logic to create week_num
        # Note: We use the alias 'as subq' because SQL Server requires it
        partition_sql = "(SELECT *, CAST(week AS INT) as week_num FROM dbo.sttwk) as subq"
        
        df = reader.option("dbtable", partition_sql) \
                   .option("partitionColumn", "week_num") \
                   .option("lowerBound", "1") \
                   .option("upperBound", "53") \
                   .option("numPartitions", "8") \
                   .load()
    else:
        # Standard load for smaller tables
        df = reader.option("dbtable", table_name).load()

    # Write to DuckDB via Arrow (the fastest 2026 method)
    con = duckdb.connect(DUCKDB_PATH)
    con.execute(f"DROP TABLE IF EXISTS {table_name}")
    con.register("temp_spark_df", df)
    con.execute(f"CREATE TABLE {table_name} AS SELECT * FROM temp_spark_df")
    
    duration = time.time() - start_time
    print(f"Migrated {table_name} in {duration:.2f} seconds using {df.rdd.getNumPartitions()} partitions.")
    
    con.close()
    spark.stop()



In [12]:
def create_final_report():
    con = duckdb.connect(DUCKDB_PATH)
    
    report_sql = """
    DROP TABLE IF EXISTS Complete_2years_Wk_Report_bySubChannel_All_Dtl_StockPoint;

    CREATE TABLE Complete_2years_Wk_Report_bySubChannel_All_Dtl_StockPoint AS 
    SELECT 
        f.region, a.branch_id, f.branch_name, a.year, a.month, a.week, 
        c.channel, c.sub_channel, 
        e.prodLine, e.prodgrp,
        d.udf6 AS psub_id, 
        g.depo, g.sp_name, g.sp_id, d.text_id, d.product, d.udf1 AS psub_id1,
        CASE WHEN c.CRM = 'Y' THEN 'Y' ELSE 'N' END AS CRM,
        CASE WHEN ((h.sls_id_chl <> '' AND y.sls_id <> '') OR h.sls_id_org <> '') AND y.no <= 100 THEN 'Y' ELSE 'N' END AS Sls_100,
        CASE WHEN (h.sls_id_chl <> '' AND y.sls_id <> '') OR h.sls_id_org <> '' THEN 'Y' ELSE 'N' END AS Sls_200,
        CASE WHEN u.custd_id <> '' THEN 'Y' ELSE 'N' END AS DW_18,
        CASE WHEN v.custd_id <> '' THEN 'Y' ELSE 'N' END AS DW_Oth,
        c.Retwhs,
        CASE WHEN c.Chiller = 'Y' THEN 'Y' ELSE 'N' END AS Chiller,
        CAST(SUM(a.net_qty) AS DECIMAL(17,5)) AS net_qty,
        CAST(SUM(a.net_amt) AS DECIMAL(17,5)) AS net_amt,
        CAST(SUM(a.grs_qty) AS DECIMAL(17,5)) AS grs_qty,
        CAST(SUM(a.grs_amt) AS DECIMAL(17,5)) AS grs_amt,
        CAST(SUM(a.rtn_qty) AS DECIMAL(17,5)) AS rtn_qty,
        CAST(SUM(a.rtn_amt) AS DECIMAL(17,5)) AS rtn_amt,
        CAST(SUM(a.grs_amt - a.net_amt - a.rtn_amt) AS DECIMAL(17,5)) AS disc_amt,
        CAST(SUM(i.disc_b_amt) AS DECIMAL(17,5)) AS Disc_B,
        CAST(SUM(j.disc_PotTunai) AS DECIMAL(17,5)) AS Disc_PT, 
        CASE WHEN x.sls_type LIKE '%canva%' THEN 'Y' ELSE 'N' END AS Canvass,
        CASE WHEN c.Mkt = 'Y' THEN 'Y' ELSE 'N' END AS Mkt
    FROM sttwk AS a
    LEFT OUTER JOIN customerdist_1910a c ON a.custd_id = c.kode_iap
    LEFT OUTER JOIN product d ON a.prod_id = d.prod_id
    LEFT OUTER JOIN prodcat e ON d.psub_id = e.psub_id
    LEFT OUTER JOIN branch f ON a.branch_id = f.branch_id
    LEFT OUTER JOIN stockpoint g ON a.sp_id = g.sp_id
    LEFT OUTER JOIN discount_b i ON a.custd_id = i.custd_id AND a.prod_id = i.prod_id AND a.year = i.year AND a.week = i.week AND a.sls_id = i.sls_id
    LEFT OUTER JOIN discount_potTunai j ON a.custd_id = j.custd_id AND a.prod_id = j.prod_id AND a.year = j.year AND a.week = j.week AND a.sls_id = j.sls_id
    LEFT OUTER JOIN sls_chlprn h ON a.branch_id = h.br_chl AND a.sp_id = h.sp_chl AND a.sls_id = h.sls_id_chl AND a.year = h.year AND a.week = h.week
    LEFT OUTER JOIN slsmanexc y ON h.br_chl = y.branch_id AND h.sp_chl = y.sp_id AND h.sls_id_prn = y.sls_id 
    LEFT OUTER JOIN slsexc_chlprn6 b ON a.sp_id = b.sp_id1 AND a.sls_id = b.sls_id1 AND a.year = b.year AND a.week = b.week
    LEFT OUTER JOIN dw_ilo_18 u ON a.custd_id = u.custd_id
    LEFT OUTER JOIN dw_ilo_oth v ON a.custd_id = v.custd_id
    LEFT OUTER JOIN slsman x ON a.branch_id = x.branch_id AND a.sp_id = x.sp_id AND a.sls_id = x.sls_id
    GROUP BY ALL;
    """
    
    start = time.time()
    con.execute(report_sql)
    end = time.time()
    
    print(f"Final report created in {end - start:.2f} seconds.")
    con.close()


In [15]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

# 1. Define configuration
default_args = {
    'owner': 'airflow',
    'start_date': datetime(2026, 1, 1),
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}

# 3. Create the DAG structure
# Note: Changed schedule_interval to schedule
with DAG('sql_server_to_duckdb_migration',
         default_args=default_args,
         schedule='@daily', 
         catchup=False) as dag:

    # Task for final report
    report_task = PythonOperator(
        task_id='create_final_report_table',
        python_callable=create_final_report
    )

    for table in TABLES_TO_COPY:
        copy_task = PythonOperator(
            task_id=f'copy_{table}',
            python_callable=migrate_table,
            op_args=[table]
        )
        
        # Dependency: migrate tables before creating report
        copy_task >> report_task

print("Airflow DAG defined successfully using modern 2026 syntax!")


C:\Users\HYPE AMD\AppData\Local\Temp\ipykernel_22012\2310528127.py:2 DeprecatedImportWarning: The `airflow.operators.python.PythonOperator` attribute is deprecated. Please use `'airflow.providers.standard.operators.python.PythonOperator'`.

Airflow DAG defined successfully using modern 2026 syntax!
